# Pixels -> millimetres (`pirouette_data.processing`)

This notebook calibrates the tracked keypoints from pixels to millimetres using the chamber
corners as a known-size reference, and appends `<bodypart>_x_mm` / `<bodypart>_y_mm` columns.

Calibration:
- **Length = 373 mm** — pooled median of the horizontal edges (`ul->ur`, `ll->lr`) -> `length_px` (x scale).
- **Width = 194 mm** — pooled median of the vertical edges (`ul->ll`, `ur->lr`) -> `width_px` (y scale).
- Coordinates are expressed relative to the chamber origin (upper-left corner `ul`), so
  `ul` -> (0, 0) mm and points are the position *inside* the chamber. The image convention is
  kept (x increases right, y increases down toward `ll`).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pirouette_data import ingestion, processing

pd.set_option("display.max_columns", 60)

## Load pose data

Chamber calibration only needs the pose keypoints, so we load a single `.h5` directly. (The same
`processing` calls work on the full combined DataFrame from `ingestion.build_dataset`.)

In [ ]:
POSE_DIR = r"C:/Users/brandon.pratt/Desktop/data/body-kinematics/pose_data"
h5_files = sorted(__import__("pathlib").Path(POSE_DIR).glob("*.h5"))

df = ingestion.load_pose_h5(h5_files[0]).reset_index()  # flattened, 'frame' column
print(f"{h5_files[0].name}: {df.shape}")
df.head(3)

## Distribution of chamber edge lengths (pixels)

Each frame contributes four edge lengths. We pool the two horizontal edges (-> length) and the two
vertical edges (-> width) and take the **median** (robust to frames where a corner is occluded).

In [ ]:
def edge(a, b):
    return np.hypot(df[f"{a}_x"] - df[f"{b}_x"], df[f"{a}_y"] - df[f"{b}_y"])


length_edges = pd.concat([edge("ul_champber", "ur_champber"), edge("ll_chamber", "lr_chamber")])
width_edges = pd.concat([edge("ul_champber", "ll_chamber"), edge("ur_champber", "lr_chamber")])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, data, label, color in (
    (axes[0], length_edges, "length (ul-ur, ll-lr)", "tab:blue"),
    (axes[1], width_edges, "width (ul-ll, ur-lr)", "tab:orange"),
):
    med = np.nanmedian(data)
    ax.hist(data.dropna(), bins=200, color=color, alpha=0.7)
    ax.axvline(med, color="k", ls="--", label=f"median = {med:.1f} px")
    ax.set(xlabel="edge length (px)", ylabel="count", title=label)
    ax.legend()
plt.tight_layout()
plt.show()

## Estimate the scale

In [ ]:
scale = processing.estimate_chamber_scale(df, likelihood_threshold=0.6)
print(scale)
print(f"\nlength: {scale.length_px:.1f} px  = 373 mm  ->  {scale.mm_per_px_x:.4f} mm/px (x)")
print(f"width : {scale.width_px:.1f} px  = 194 mm  ->  {scale.mm_per_px_y:.4f} mm/px (y)")
print(f"origin (ul): ({scale.origin_px_x:.1f}, {scale.origin_px_y:.1f}) px")
print(f"\npixel aspect (len/wid): {scale.length_px / scale.width_px:.3f}   mm aspect: {373 / 194:.3f}")

## Append millimetre columns

In [ ]:
out = processing.append_mm_columns(df, scale=scale)
mm_cols = [c for c in out.columns if c.endswith("_mm")]
print("new columns:", mm_cols)
out[["left_ear_x", "left_ear_x_mm", "left_ear_y", "left_ear_y_mm"]].head(3)

In [ ]:
# Median corner positions in mm should sit near the ideal rectangle.
for c in ["ul_champber", "ur_champber", "lr_chamber", "ll_chamber"]:
    print(f"{c:14s} ({np.nanmedian(out[c + '_x_mm']):7.1f}, {np.nanmedian(out[c + '_y_mm']):7.1f}) mm")
print("ideal          ul(0,0) ur(373,0) lr(373,194) ll(0,194)")

## Visualise: keypoints inside the chamber (mm)

In [ ]:
# Chamber outline from median corner positions
corners = ["ul_champber", "ur_champber", "lr_chamber", "ll_chamber", "ul_champber"]
cx = [np.nanmedian(out[f"{c}_x_mm"]) for c in corners]
cy = [np.nanmedian(out[f"{c}_y_mm"]) for c in corners]

# Sample the hat keypoint (head marker) where it is confidently tracked
conf = out["hat_likelihood"] > 0.6
hx = out.loc[conf, "hat_x_mm"].to_numpy()[::100]
hy = out.loc[conf, "hat_y_mm"].to_numpy()[::100]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(cx, cy, "k-", lw=2, label="chamber")
ax.scatter(hx, hy, s=4, alpha=0.3, color="tab:red", label="hat")
ax.scatter([0], [0], color="k", zorder=5)
ax.annotate("ul (origin)", (0, 0), textcoords="offset points", xytext=(6, 6))
ax.set(xlabel="x (mm)", ylabel="y (mm)", title="Tracked hat position in chamber-mm coordinates")
ax.invert_yaxis()  # image convention: y increases downward
ax.set_aspect("equal")
ax.legend()
plt.tight_layout()
plt.show()